# Практична робота №4 — Дерева рішень для класифікації (Coffee & Health) — FIXED
**Датасет:** `synthetic_coffee_health_10000.csv`  
**Ціль:** `Health_Issues` ∈ {None, Mild, Moderate, Severe}

**Що є в ноутбуці:**
- Завантаження датасету, підготовка ознак (one‑hot), ціль — LabelEncoding
- Розбиття Train/Val/Test (70/10/20 зі стратифікацією)
- **Власне дерево рішень (Gini)** + **feature importance**
- **Прунінг** за валідацією (α‑регуляризація)
- Порівняння зі `sklearn.DecisionTreeClassifier` та `RandomForestClassifier`
- Метрики (Accuracy, Macro‑F1, MCC), матриці плутанини, бар‑чарти важливостей
- 5‑fold **крос‑валідація** для sklearn‑моделей

> Версія FIXED: змінні спільні між клітинками, назви фіч збережено в `FEATURE_NAMES`, 
> порядок виконання — зверху вниз без помилок `NameError`.

In [ ]:
# === 0) Імпорти (без pip install) ===========================================
import os, re, math, time, json, random, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from typing import Optional, Dict, Any

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, matthews_corrcoef
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

print("✅ Імпорти готові")

In [ ]:
# === 1) Завантаження датасету ================================================
CANDIDATES = [
    "/mnt/data/synthetic_coffee_health_10000.csv",
    "synthetic_coffee_health_10000.csv",
    "/content/synthetic_coffee_health_10000.csv"
]

DATA_PATH = None
for p in CANDIDATES:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    try:
        from google.colab import files
        print("📥 Завантажте synthetic_coffee_health_10000.csv")
        uploaded = files.upload()
        if uploaded:
            DATA_PATH = next(iter(uploaded.keys()))
    except Exception:
        pass

if DATA_PATH is None:
    raise FileNotFoundError("❌ synthetic_coffee_health_10000.csv не знайдено. Покладіть файл поруч або завантажте в Colab.")

print("📄 Файл:", DATA_PATH)
DF_RAW = pd.read_csv(DATA_PATH)
print("✅ Розмір:", DF_RAW.shape)
display(DF_RAW.head())

In [ ]:
# === 2) Підготовка даних =====================================================
# Ціль — Health_Issues. Видаляємо ID‑поля, кодуємо категоріальні — one‑hot.

# 2.1 Знаходимо ціль
TARGET_COL = None
for c in DF_RAW.columns:
    if c.strip().lower() == "health_issues":
        TARGET_COL = c
        break
if TARGET_COL is None:
    raise ValueError("❌ Не знайдено цільову колонку 'Health_Issues'.")

# 2.2 Викидаємо ID‑стовпці
drop_ids = [c for c in DF_RAW.columns if ("id" in c.strip().lower()) and (c != TARGET_COL)]
DF = DF_RAW.drop(columns=drop_ids) if drop_ids else DF_RAW.copy()

# 2.3 Розділяємо X / y
y_series = DF[TARGET_COL].astype(str)
X_df = DF.drop(columns=[TARGET_COL])

# 2.4 Кодування: числові пропуски → медіана, категоріальні → "Unknown"
cat_cols = [c for c in X_df.columns if X_df[c].dtype == 'object']
num_cols = [c for c in X_df.columns if c not in cat_cols]

for c in num_cols:
    if X_df[c].isna().any():
        X_df[c] = X_df[c].fillna(X_df[c].median())
for c in cat_cols:
    if X_df[c].isna().any():
        X_df[c] = X_df[c].fillna("Unknown")

# 2.5 One‑hot
X_dummies = pd.get_dummies(X_df, columns=cat_cols, drop_first=False)
FEATURE_NAMES = X_dummies.columns.tolist()

# 2.6 LabelEncoding цілі
LE = LabelEncoder()
y_enc = LE.fit_transform(y_series)

print(f"✅ Після препроцесингу: X={X_dummies.shape}, y={y_enc.shape}, класи={list(LE.classes_)}")
display(X_dummies.head())

In [ ]:
# === 3) Train / Val / Test ===================================================
# 70% train_full, 10% val, 20% test
X_full = X_dummies.values
y_full = y_enc

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_full, y_full, test_size=0.20, random_state=42, stratify=y_full
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.125, random_state=42, stratify=y_train_full
)  # 0.125 від 80% ≈ 10%

print("✅ Розбиття:", X_train.shape, X_val.shape, X_test.shape)

In [ ]:
# === 4) Власне дерево рішень (Gini) + feature importance ====================
class MyDecisionTree:
    def __init__(self, max_depth=8, min_samples_split=10, min_samples_leaf=5):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.n_total = None
        self.tree = None
        self.feature_importance_ = None

    @staticmethod
    def gini(y):
        if len(y) == 0:
            return 0.0
        _, counts = np.unique(y, return_counts=True)
        p = counts / counts.sum()
        return 1.0 - np.sum(p ** 2)

    def _best_split(self, X, y):
        n_samples, n_features = X.shape
        parent_gini = self.gini(y)

        best_feat, best_thr = None, None
        best_weighted = float('inf')
        best_left_idx, best_right_idx = None, None

        for j in range(n_features):
            col = X[:, j]
            uniq = np.unique(col)
            if len(uniq) <= 2:
                thresholds = [0.5]
            else:
                # обмежена кількість порогів (до 9 квантилів) — швидко і стабільно
                qs = np.linspace(0.1, 0.9, 9)
                thresholds = np.unique(np.quantile(col, qs))

            for thr in thresholds:
                left_idx  = np.nonzero(col <= thr)[0]
                right_idx = np.nonzero(col >  thr)[0]

                if len(left_idx) < self.min_samples_leaf or len(right_idx) < self.min_samples_leaf:
                    continue

                g_left  = self.gini(y[left_idx])
                g_right = self.gini(y[right_idx])
                w = (len(left_idx)/n_samples)*g_left + (len(right_idx)/n_samples)*g_right

                if w < best_weighted:
                    best_weighted = w
                    best_feat, best_thr = j, float(thr)
                    best_left_idx, best_right_idx = left_idx, right_idx

        if best_feat is None:
            return None, None, None, None, None

        impurity_decrease = parent_gini - best_weighted
        return best_feat, best_thr, best_left_idx, best_right_idx, impurity_decrease

    def _build(self, X, y, depth=0):
        node = {
            "leaf": False,
            "n": len(y),
            "gini": self.gini(y),
            "class": int(np.bincount(y).argmax()),
        }

        # зупинка
        if depth >= self.max_depth or len(y) < self.min_samples_split or node["gini"] == 0.0:
            node["leaf"] = True
            return node

        feat, thr, left_idx, right_idx, dec = self._best_split(X, y)
        if feat is None:
            node["leaf"] = True
            return node

        node["feature"] = feat
        node["threshold"] = thr
        node["impurity_decrease"] = float(dec)

        # важливість ознаки (additive, зважено на частку зразків у вузлі)
        self.feature_importance_[feat] += (len(y)/self.n_total) * dec

        node["left"]  = self._build(X[left_idx],  y[left_idx],  depth+1)
        node["right"] = self._build(X[right_idx], y[right_idx], depth+1)
        return node

    def fit(self, X, y):
        self.n_total = len(y)
        self.feature_importance_ = np.zeros(X.shape[1], dtype=float)
        self.tree = self._build(X, y, depth=0)
        s = self.feature_importance_.sum()
        if s > 0:
            self.feature_importance_ /= s
        return self

    def _predict_one(self, x, node):
        if node.get("leaf", False):
            return node["class"]
        if x[node["feature"]] <= node["threshold"]:
            return self._predict_one(x, node["left"])
        else:
            return self._predict_one(x, node["right"])

    def predict(self, X):
        return np.array([self._predict_one(x, self.tree) for x in X])

    # ---------------- ПРУНІНГ за валідацією (α‑regularized) ------------------
    def _node_leaf_error(self, y_true, major_cls):
        if len(y_true) == 0:
            return 0.0
        return 1.0 - (np.sum(y_true == major_cls) / len(y_true))

    def _prune_rec(self, node, X_val, y_val, idxs, alpha):
        if node.get("leaf", False):
            return self._node_leaf_error(y_val[idxs], node["class"]) + alpha

        feat, thr = node["feature"], node["threshold"]
        mask_left = X_val[idxs, feat] <= thr
        left_idx  = idxs[mask_left]
        right_idx = idxs[~mask_left]

        left_cost  = self._prune_rec(node["left"],  X_val, y_val, left_idx,  alpha)
        right_cost = self._prune_rec(node["right"], X_val, y_val, right_idx, alpha)
        subtree_cost = left_cost + right_cost

        as_leaf_cost = self._node_leaf_error(y_val[idxs], node["class"]) + alpha

        if as_leaf_cost <= subtree_cost:
            node.clear()
            node.update({"leaf": True, "class": node.get("class", 0), "n": len(idxs)})
            return as_leaf_cost
        else:
            return subtree_cost

    def prune(self, X_val, y_val, alpha=0.0):
        idxs = np.arange(len(y_val))
        self._prune_rec(self.tree, X_val, y_val, idxs, alpha)

        # переобчислюємо feature_importance_
        self.feature_importance_[:] = 0.0
        self._accumulate_importance(self.tree)
        s = self.feature_importance_.sum()
        if s > 0:
            self.feature_importance_ /= s
        return self

    def _accumulate_importance(self, node):
        if node.get("leaf", False):
            return
        feat = node.get("feature")
        gain = node.get("impurity_decrease", 0.0)
        n    = node.get("n", 0)
        if feat is not None and n:
            self.feature_importance_[feat] += (n / self.n_total) * gain
        self._accumulate_importance(node["left"])
        self._accumulate_importance(node["right"])

In [ ]:
# === 5) Навчання, прунінг, порівняння з sklearn ==============================
def evaluate_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="macro")
    mcc = matthews_corrcoef(y_true, y_pred)
    cm  = confusion_matrix(y_true, y_pred)
    return {"model": name, "accuracy": acc, "macro_f1": f1, "mcc": mcc, "cm": cm}

# 5.1 Моє дерево (без прунінгу)
my_dt = MyDecisionTree(max_depth=10, min_samples_split=20, min_samples_leaf=10).fit(X_train, y_train)
y_pred_test = my_dt.predict(X_test)
res_my = evaluate_model("MyDecisionTree (raw)", y_test, y_pred_test)

# 5.2 Прунінг: підбір α по валідації
best_alpha, best_acc = 0.0, -1.0
best_model_after_prune = None

for alpha in [0.0, 0.001, 0.01, 0.05, 0.1]:
    pruned = copy.deepcopy(my_dt)
    pruned.prune(X_val, y_val, alpha=alpha)
    y_pred_val = pruned.predict(X_val)
    acc_val = accuracy_score(y_val, y_pred_val)
    if acc_val > best_acc:
        best_acc = acc_val
        best_alpha = alpha
        best_model_after_prune = pruned

y_pred_test_pruned = best_model_after_prune.predict(X_test)
res_my_pruned = evaluate_model(f"MyDecisionTree (pruned α={best_alpha})", y_test, y_pred_test_pruned)

# 5.3 sklearn дерева
dt_sk = DecisionTreeClassifier(criterion="gini", max_depth=10, min_samples_leaf=10, random_state=42)
dt_sk.fit(X_train, y_train)
y_pred_dt = dt_sk.predict(X_test)
res_dt = evaluate_model("sklearn DecisionTree", y_test, y_pred_dt)

rf = RandomForestClassifier(n_estimators=200, criterion="gini", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
res_rf = evaluate_model("RandomForest", y_test, y_pred_rf)

# 5.4 Зведена таблиця
ROWS = [res_my, res_my_pruned, res_dt, res_rf]
METRICS_DF = pd.DataFrame([
    {"Model": r["model"], "Accuracy": r["accuracy"], "MacroF1": r["macro_f1"], "MCC": r["mcc"]}
    for r in ROWS
]).sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

print("✅ Метрики на тесті:")
display(METRICS_DF)
print("🔥 Найкраща модель:", METRICS_DF.iloc[0]["Model"])

In [ ]:
# === 6) Візуалізації: матриці плутанини та важливості ознак ==================
def plot_cm(cm, title):
    plt.figure()
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")
    plt.tight_layout()
    plt.show()

# 6.1 Матриці плутанини для всіх моделей
for r in ROWS:
    plot_cm(r["cm"], r["model"])

# 6.2 Важливість ознак (моє дерево після прунінгу) — топ‑15
imp_my = best_model_after_prune.feature_importance_
top_idx = np.argsort(imp_my)[::-1][:15]
top_names = [FEATURE_NAMES[i] for i in top_idx]

plt.figure()
plt.bar(range(len(top_idx)), imp_my[top_idx])
plt.xticks(range(len(top_idx)), top_names, rotation=90)
plt.title("MyDecisionTree Feature Importance (top-15)")
plt.tight_layout()
plt.show()

# 6.3 Важливість ознак RandomForest — топ‑15
rf_imp = rf.feature_importances_
rf_top = np.argsort(rf_imp)[::-1][:15]
rf_names = [FEATURE_NAMES[i] for i in rf_top]

plt.figure()
plt.bar(range(len(rf_top)), rf_imp[rf_top])
plt.xticks(range(len(rf_top)), rf_names, rotation=90)
plt.title("RandomForest Feature Importance (top-15)")
plt.tight_layout()
plt.show()

In [ ]:
# === 7) 5‑fold крос‑валідація для sklearn‑моделей ===========================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

dt_cv = DecisionTreeClassifier(criterion="gini", max_depth=10, min_samples_leaf=10, random_state=42)
rf_cv = RandomForestClassifier(n_estimators=200, criterion="gini", random_state=42, n_jobs=-1)

acc_dt = cross_val_score(dt_cv, X_train_full, y_train_full, cv=skf, scoring="accuracy")
acc_rf = cross_val_score(rf_cv, X_train_full, y_train_full, cv=skf, scoring="accuracy")

print("DecisionTree 5‑fold Accuracy:", np.round(acc_dt, 4), "→ mean:", round(acc_dt.mean(), 4))
print("RandomForest 5‑fold Accuracy:", np.round(acc_rf, 4), "→ mean:", round(acc_rf.mean(), 4))

## Підсумки
- Реалізовано **власне дерево рішень** з критерієм Gini + підрахунок **feature importance**.
- Додано **прунінг** за валідаційною множиною (α‑regularized), підбір α.
- Порівняно з `sklearn.DecisionTreeClassifier` та `RandomForestClassifier` на Test, подано **Accuracy, Macro‑F1, MCC** і **матриці плутанини**.
- Побудовано бар‑чарти важливих ознак та виконано 5‑fold **крос‑валідацію**.